### sql using python API

In [0]:
df = spark.sql("select * from workspace.default.movies where studio= 'Marvel Studios'")
df.show()

# sql using %sql

In [0]:
%sql

select * from workspace.default.movies where studio= 'Marvel Studios'

###Join Operations

In [0]:
from pyspark.sql import functions as F, types as T

rows_customers = [
    (1,  "Asha",  "IN", True),
    (2,  "Bob",   "US", False),
    (3,  "Chen",  "CN", True),
    (4,  "Diana", "US", None),
    (None, "Ghost","UK", False),     # NULL key to demo null join behavior
]

rows_orders = [
    (101, 1,   120.0, "IN"),
    (102, 1,    80.0, "IN"),
    (103, 2,    50.0, "US"),
    (104, 5,    30.0, "DE"),         # no matching customer_id
    (105, 3,   200.0, "CN"),
    (106, None, 15.0, "UK"),         # NULL key won’t match
    (107, 3,    40.0, "CN"),
    (108, 2,    75.0, "US"),
]

schema_customers = T.StructType([
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("name",        T.StringType(),  True),
    T.StructField("country",     T.StringType(),  True),
    T.StructField("vip",         T.BooleanType(), True),
])

schema_orders = T.StructType([
    T.StructField("order_id",    T.IntegerType(), True),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("amount",      T.DoubleType(),  True),
    T.StructField("country",     T.StringType(),  True),  # same column name to show collisions
])

df_customers = spark.createDataFrame(rows_customers,schema_customers)

df_orders = spark.createDataFrame(rows_orders,schema_orders)

In [0]:
df_customers.display()
df_orders.display()

In [0]:
df_orders.join(df_customers,on='customer_id',how="inner").display()

In [0]:
df_orders.join(df_customers,on='customer_id',how="left").display()

In [0]:
df_orders.join(df_customers,on='customer_id',how="right").display()

In [0]:
o = df_orders.alias('o')
c = df_customers.alias('c')

# NOTE: aliasing helps to avoid duplicate columns in join operations like duplicate country column in below query

In [0]:
o.join(c,on='customer_id',how="inner").display()

In [0]:
df_clean_inner = (
     o.join(c,on='customer_id',how="inner")
     .select("customer_id","amount","customer_id",
             F.col("o.country").alias("Shipment_country"),
             "name",
             F.col("c.country").alias("Customer_country"),
             "vip"
             )
     )
df_clean_inner.display()

In [0]:
o.join(c,on='customer_id',how="fullouter").display()

In [0]:
o.join(c,on='customer_id',how="left_semi").display()  #orders with known cutomers(non null customer_id join key) 

In [0]:
o.join(c,on='customer_id',how="left_anti").display()  #orders without any customer information in customer table

In [0]:
o.join(c,on=['customer_id','country'],how="inner").display()